In [1]:
import numpy as np
import gymnasium as gym

# --- 1. Environment Setup ---
# Deterministic FrozenLake-v1 (is_slippery=False)
env = gym.make('FrozenLake-v1', is_slippery=False)
n_states = env.observation_space.n
n_actions = env.action_space.n

# Identify terminal states from the map description
desc = env.unwrapped.desc.astype(str)
terminal_states = [i * 4 + j for i in range(4) for j in range(4) if desc[i, j] in ['H', 'G']]

In [2]:
# --- Helper: Epsilon-Greedy Policy ---
def epsilon_greedy_policy(state, Q, epsilon):
    if np.random.rand() < epsilon:
        return np.random.choice(n_actions)
    else:
        # Break ties randomly to avoid getting stuck
        best_actions = np.argwhere(Q[state] == np.max(Q[state])).flatten()
        return np.random.choice(best_actions)

In [3]:
# --- 2. Monte Carlo Policy Control (First-Visit) ---
def monte_carlo(env, episodes=5000, gamma=0.99, epsilon=0.1):
    Q = np.zeros((n_states, n_actions))
    returns_sum = np.zeros((n_states, n_actions))
    returns_count = np.zeros((n_states, n_actions))

    for _ in range(episodes):
        episode = []
        state, _ = env.reset()
        done = False
        
        # Generate an episode
        while not done:
            action = epsilon_greedy_policy(state, Q, epsilon)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            episode.append((state, action, reward))
            state = next_state
            
        # First-Visit update
        G = 0
        visited_sa = set()
        for t in reversed(range(len(episode))):
            s, a, r = episode[t]
            G = gamma * G + r
            if (s, a) not in visited_sa:
                visited_sa.add((s, a))
                returns_sum[s, a] += G
                returns_count[s, a] += 1
                Q[s, a] = returns_sum[s, a] / returns_count[s, a]
    return Q


In [4]:
# --- 3. SARSA ---
def sarsa(env, episodes=5000, alpha=0.1, gamma=0.99, epsilon=0.1):
    Q = np.zeros((n_states, n_actions))
    
    for _ in range(episodes):
        state, _ = env.reset()
        action = epsilon_greedy_policy(state, Q, epsilon)
        done = False
        
        while not done:
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            next_action = epsilon_greedy_policy(next_state, Q, epsilon)
            
            # Update Q-value
            if done:
                td_target = reward
            else:
                td_target = reward + gamma * Q[next_state, next_action]
                
            td_error = td_target - Q[state, action]
            Q[state, action] += alpha * td_error
            
            state, action = next_state, next_action
    return Q

In [5]:
# --- 4. Policy Visualization ---
def print_policy(Q, title):
    arrows = {0: '←', 1: '↓', 2: '→', 3: '↑'}
    policy_grid = []
    
    print(f"\n{title}:")
    for i in range(4):
        row = []
        for j in range(4):
            state = i * 4 + j
            if state in terminal_states:
                row.append('T')
            else:
                best_action = np.argmax(Q[state])
                row.append(arrows[best_action])
        print(" ".join(row))
        policy_grid.append(row)

In [6]:
def evaluate_policy(env, Q, eval_episodes=1000):
    """
    Evaluates the learned Q-table by running a purely greedy policy.
    Returns the success rate (percentage of episodes where the goal is reached).
    """
    success_count = 0
    
    for _ in range(eval_episodes):
        state, _ = env.reset()
        done = False
        
        while not done:
            # Always choose the best known action (greedy, epsilon=0)
            action = np.argmax(Q[state])
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # In FrozenLake, reaching the goal yields a reward of 1.0
            if reward == 1.0:
                success_count += 1
                
            state = next_state
            
    success_rate = (success_count / eval_episodes) * 100
    return success_rate

In [7]:
# --- Execution & Evaluation ---
if __name__ == "__main__":
    eval_runs = 1000  # Number of episodes to evaluate the final policy

    print("--- Monte Carlo ---")
    print("Training Monte Carlo...")
    Q_mc = monte_carlo(env, episodes=10000, epsilon=0.2)
    print_policy(Q_mc, "Monte Carlo Policy")
    
    # Evaluate Monte Carlo
    mc_success_rate = evaluate_policy(env, Q_mc, eval_episodes=eval_runs)
    print(f"Monte Carlo Evaluation Success Rate: {mc_success_rate:.2f}% over {eval_runs} episodes\n")

    print("--- SARSA ---")
    print("Training SARSA...")
    Q_sarsa = sarsa(env, episodes=10000, alpha=0.1, epsilon=0.2)
    print_policy(Q_sarsa, "SARSA Policy")
    
    # Evaluate SARSA
    sarsa_success_rate = evaluate_policy(env, Q_sarsa, eval_episodes=eval_runs)
    print(f"SARSA Evaluation Success Rate: {sarsa_success_rate:.2f}% over {eval_runs} episodes")

--- Monte Carlo ---
Training Monte Carlo...

Monte Carlo Policy:
↓ ← ↑ ←
↓ T ↓ T
→ ↓ ↓ T
T → → T
Monte Carlo Evaluation Success Rate: 100.00% over 1000 episodes

--- SARSA ---
Training SARSA...

SARSA Policy:
↓ ← ← ←
↓ T ↑ T
→ → ↓ T
T → → T
SARSA Evaluation Success Rate: 100.00% over 1000 episodes
